# LLM Bias Detection v2 — Inference on forced choice responses
runs trained BERT and RoBERTa and T5 classifiers on the justificatio` text from GPT, Claude,
and Llama forced-choice responses 

Tests whether the underlying justification text leans left/right/center REGARDLESS of
the LLM's stated choice (A/B/neutral)



In [ ]:

! pip install transformers scikit-learn sentencepiece 

In [ ]:

import os
import pandas as pd
import torch
from transformers import(BertTokenizer, BertForSequenceClassification,AutoTokenizer, AutoModelForSequenceClassification,T5Tokenizer, T5ForConditionalGeneration)
import warnings
warnings.filterwarnings('ignore')

In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# file paths
# LLM response csv fies
DATA_PATHS = {
    'gpt_us':'/kaggle/input/llm-responses-v2/gpt_us.csv',
    'gpt_uk':'/kaggle/input/llm-responses-v2/gpt_uk.csv',
    'claude_us':'/kaggle/input/llm-responses-v2/claude_us.csv',
    'claude_uk':'/kaggle/input/llm-responses-v2/claude_uk.csv',
    'llama_us':'/kaggle/input/llm-responses-v2/llama_us.csv',
    'llama_uk':'/kaggle/input/llm-responses-v2/llama_uk.csv',
}
# Domain-specific political models (PolitBERT, POLITICS) + generic T5 baseline
MODEL_PATHS = {
    'politbert_us':'/kaggle/input/thesis-models/politbert_us',
    'politbert_uk':'/kaggle/input/thesis-models/politbert_uk',
    'politics_us':'/kaggle/input/thesis-models/politics_us',
    'politics_uk':'/kaggle/input/thesis-models/politics_uk',
    't5_us':'/kaggle/input/thesis-models/t5_us',
    't5_uk':'/kaggle/input/thesis-models/t5_uk',
}
OUTPUT_DIR ='/kaggle/working/bias_results_v2'

In [ ]:
# label mapping
US_ID2LABEL = {0: 'left', 1: 'right'}
UK_ID2LABEL = {0: 'left', 1: 'center', 2: 'right'}
US_VALID_LABELS =['left', 'right']
UK_VALID_LABELS= ['left', 'center', 'right']
MAX_LENGTH= 256

In [ ]:
# load llm response CSVs

llm_data ={}
for name, path in DATA_PATHS.items():
    df= pd.read_csv(path)
    df.columns = [c.strip().lower() for c in df.columns]
    # the text to classify is the 'justification' column (not 'answer'/'response')
    df= df.dropna(subset=['justification']).reset_index(drop=True)
    df['justification'] =df['justification'].astype(str)
    df['choice'] = df['choice'].astype(str).str.strip()
    llm_data[name]= df

In [ ]:
#PolitBERT inference function
def politbert_predict(texts, model_path, id2label, max_length=256):
    print(f'Loading PolitBERT from {model_path}')
    tokenizer= BertTokenizer.from_pretrained(model_path)
    model= BertForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()
    predictions =[]
    confidences =[]
    with torch.no_grad():
        for text in texts:
            enc = tokenizer(str(text), max_length=max_length, padding='max_length',truncation=True, return_tensors='pt')
            input_ids= enc['input_ids'].to(device)
            attention_mask = enc['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs= torch.softmax(outputs.logits, dim=1)
            pred_id =torch.argmax(probs, dim=1).item()
            conf= probs[0][pred_id].item()
            predictions.append(id2label[pred_id])
            confidences.append(round(conf, 4))
    return predictions,confidences

In [ ]:
#POLITICS inference function
def politics_predict(texts, model_path, id2label, max_length=256):
    print(f'Loading POLITICS from {model_path}')
    tokenizer= AutoTokenizer.from_pretrained(model_path)
    model= AutoModelForSequenceClassification.from_pretrained(model_path)
    model.to(device)
    model.eval()
    predictions =[]
    confidences =[]
    with torch.no_grad():
        for text in texts:
            enc= tokenizer(
                str(text), max_length=max_length, padding='max_length',truncation=True, return_tensors='pt')
            input_ids= enc['input_ids'].to(device)
            attention_mask= enc['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs= torch.softmax(outputs.logits, dim=1)
            pred_id =torch.argmax(probs, dim=1).item()
            conf= probs[0][pred_id].item()
            predictions.append(id2label[pred_id])
            confidences.append(round(conf, 4))
    return predictions, confidences
print('POLITICS inference function defined')

In [ ]:
#T5 inference function
# T5 is text-to-text: input is prefixed, output is generated text decoded to a label
def t5_predict(texts, model_path, valid_labels, max_length=256):
    print(f'Loading T5 from {model_path}')
    tokenizer= T5Tokenizer.from_pretrained(model_path)
    model=T5ForConditionalGeneration.from_pretrained(model_path)
    model.to(device)
    model.eval()
    predictions =[]
    confidences = []#T5 doesn't give a softmax confidence directly, recorded as n/a
    with torch.no_grad():
        for text in texts:
            input_text= f'classify political ideology: {str(text)}'
            enc =tokenizer(input_text,max_length=max_length, padding='max_length',truncation=True, return_tensors='pt')
            input_ids = enc['input_ids'].to(device)
            attention_mask = enc['attention_mask'].to(device)
            generated= model.generate(input_ids=input_ids,attention_mask=attention_mask,max_new_tokens=5)
            pred =tokenizer.decode(generated[0], skip_special_tokens=True).strip().lower()
            if pred not in valid_labels:
                pred = valid_labels[0]# default fallback if T5 generates something unexpected
            predictions.append(pred)
            confidences.append('N/A')
    return predictions, confidences
print('T5 inference function defined')

In [ ]:
#run inference for US LLM justifications
us_results =[]
for llm_name in ['gpt_us','claude_us','llama_us']:
    df= llm_data[llm_name]
    texts= df['justification'].tolist()
    llm = llm_name.replace('_us', '')
    politbert_preds, politbert_confs = politbert_predict(texts,MODEL_PATHS['politbert_us'], US_ID2LABEL)
    politics_preds,  politics_confs = politics_predict(texts,MODEL_PATHS['politics_us'], US_ID2LABEL)
    t5_preds,t5_confs = t5_predict(texts,MODEL_PATHS['t5_us'],US_VALID_LABELS)
    for i, row in df.iterrows():
        us_results.append({
            'llm':llm,
            'country':'US',
            'question':row['question'],
            'choice':row['choice'],
            'justification':row['justification'],
            'politbert_label':politbert_preds[i],
            'politbert_confidence': politbert_confs[i],
            'politics_label':politics_preds[i],
            'politics_confidence': politics_confs[i],
            't5_label':t5_preds[i],
            't5_confidence':t5_confs[i],
        })
us_df= pd.DataFrame(us_results)

In [ ]:
#run inference for UK LLM justifications
uk_results =[]
for llm_name in ['gpt_uk','claude_uk','llama_uk']:
    df= llm_data[llm_name]
    texts= df['justification'].tolist()
    llm = llm_name.replace('_uk', '')
    politbert_preds, politbert_confs = politbert_predict(texts,MODEL_PATHS['politbert_uk'], UK_ID2LABEL)
    politics_preds,  politics_confs = politics_predict(texts,MODEL_PATHS['politics_uk'], UK_ID2LABEL)
    t5_preds,t5_confs = t5_predict(texts,MODEL_PATHS['t5_uk'],UK_VALID_LABELS)
    for i, row in df.iterrows():
        uk_results.append({
            'llm':llm,
            'country': 'UK',
            'question':row['question'],
            'choice':row['choice'],
            'justification':row['justification'],
            'politbert_label':politbert_preds[i],
            'politbert_confidence': politbert_confs[i],
            'politics_label': politics_preds[i],
            'politics_confidence': politics_confs[i],
            't5_label':t5_preds[i],
            't5_confidence':t5_confs[i],
        })
uk_df= pd.DataFrame(uk_results)

In [ ]:
# combine and save results
all_results =pd.concat([us_df, uk_df], ignore_index=True)
all_results.to_csv(os.path.join(OUTPUT_DIR, 'bias_results_v2_full.csv'), index=False)

In [ ]:
#neautrality per llm
for country in ['US','UK']:
    print(f'\n {country}')
    subset = all_results[all_results['country']==country]
    for llm in subset['llm'].unique():
        llm_subset =subset[subset['llm']== llm]
        counts= llm_subset['choice'].value_counts()
        pcts= llm_subset['choice'].value_counts(normalize=True).round(3) *100
        print(f'{llm.upper():8} |',end='')
        for label in counts.index:
            print(f'{label}: {counts[label]}({pcts[label]:.1f}%)',end='')
        print()